# 05b — CIB Consistency Analysis

Find **internally consistent configurations** of driver manifestations using the Weimer-Jehle balance algorithm adapted to driver-to-driver CIB scores.

A configuration is a fixed point when no single driver would "prefer" a different manifestation given what all other drivers are doing.

- Input: `morphbox_state.json` + `cib_state.json`
- Output: `consistency_state.json` (consistent configurations = scenario seeds)
- **0 API calls** — pure algorithmic step

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from src.models.morphological import DriverManifestation, MorphologicalBox
from src.models.drivers import TechDriver
from src.pipeline.morphological import (
    find_consistent_configs,
    select_scenario_seeds,
    infer_scenario_type,
)

with open("../data/outputs/morphbox_state.json") as f:
    morphbox_raw = json.load(f)

with open("../data/outputs/cib_state.json") as f:
    cib_state = json.load(f)

with open("../data/outputs/merge_state.json") as f:
    merge_state = json.load(f)

morph_box = MorphologicalBox(
    drivers=morphbox_raw["drivers"],
    manifestations=morphbox_raw["manifestations"],
    all_manifestations=[DriverManifestation(**m) for m in morphbox_raw["all_manifestations"]],
)

drivers = [TechDriver(**d) for d in merge_state["unified_drivers"]]
driver_lookup = {d.id: d for d in drivers}
manif_lookup = {m.id: m for m in morph_box.all_manifestations}

cib_matrix = cib_state["matrix"]
driver_index = {did: i for i, did in enumerate(cib_state["driver_ids"])}

print(f"Morphological box: {len(morph_box.drivers)} drivers × {len(morph_box.all_manifestations)} manifestations")
print(f"CIB matrix: {len(cib_matrix)}×{len(cib_matrix[0])}")

## Find Consistent Configurations

Random-restart hill-climbing with 5000 restarts. Each restart picks a random configuration and iterates to a fixed point.

In [ ]:
all_configs = find_consistent_configs(
    morph_box, cib_matrix, driver_index, n_restarts=5000, seed=42
)

print(f"Found {len(all_configs)} unique consistent configurations\n")
for i, cfg in enumerate(all_configs):
    stype = infer_scenario_type(cfg.configuration, morph_box)
    labels = [manif_lookup[cfg.configuration[d]].label for d in morph_box.drivers]
    print(f"Config {i+1} (score={cfg.consistency_score:.2f}, type={stype}):")
    for d_id, m_id in cfg.configuration.items():
        m = manif_lookup[m_id]
        d_name = driver_lookup[d_id].name[:35] if d_id in driver_lookup else d_id[:12]
        print(f"  {d_name:<36} → {m.label}")
    print()

## Select Diverse Scenario Seeds

Pick 4–6 seeds that differ in at least 3 drivers from each other.

In [ ]:
seeds = select_scenario_seeds(all_configs, morph_box, n=6, min_hamming=3)

print(f"Selected {len(seeds)} scenario seeds:\n")
for i, seed in enumerate(seeds):
    stype = infer_scenario_type(seed.configuration, morph_box)
    print(f"Seed {i+1}: {stype.upper()} (consistency={seed.consistency_score:.2f})")
    for d_id, m_id in seed.configuration.items():
        m = manif_lookup[m_id]
        d_name = driver_lookup[d_id].name[:35] if d_id in driver_lookup else d_id[:12]
        print(f"  {d_name:<36} → {m.label}")
    print()

In [ ]:
consistency_state = {
    "configs": [
        {
            "id": cfg.id,
            "configuration": cfg.configuration,
            "consistency_score": cfg.consistency_score,
            "is_consistent": cfg.is_consistent,
            "scenario_type": infer_scenario_type(cfg.configuration, morph_box),
        }
        for cfg in seeds
    ],
    "all_fixed_points": len(all_configs),
    "algorithm_stats": {
        "random_restarts": 5000,
        "unique_fixed_points": len(all_configs),
        "selected_count": len(seeds),
    },
}

with open("../data/outputs/consistency_state.json", "w") as f:
    json.dump(consistency_state, f, indent=2)

print(f"Saved {len(seeds)} scenario seeds to consistency_state.json")